# 3. TRAINING & EMBEDDING DENGAN INDOBERT (V2)

**Ekstraksi Kata Kunci dengan N-Gram dan IndoBERT untuk Rekomendasi Wisata Bogor**

---

## Perubahan dari V1:
- Menggunakan `data_preprocessed_indobert.csv` (TANPA stopword removal)
- IndoBERT membutuhkan konteks penuh untuk pemahaman semantik yang lebih baik

In [35]:
!pip install torch transformers sentence-transformers tqdm -q


[notice] A new release of pip available: 22.2.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = './data/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

env_info = pd.DataFrame({
    'Parameter': ['Device', 'CUDA Available'],
    'Nilai': [str(device), torch.cuda.is_available()]
})
print("TABEL: ENVIRONMENT")
display(env_info)

TABEL: ENVIRONMENT


,Parameter,Nilai
0,Device,cpu
1,CUDA Available,False


In [37]:
# Load data (TANPA stopword removal - untuk konteks semantik penuh)
df = pd.read_csv(f'{DATA_PATH}data_preprocessed_indobert.csv')

# Handle NaN - PENTING!
df['deskripsi_clean'] = df['deskripsi_clean'].fillna('').astype(str)
# Filter empty strings
df = df[df['deskripsi_clean'].str.strip() != ''].reset_index(drop=True)

data_info = pd.DataFrame({
    'Keterangan': ['Total Data Valid', 'Jumlah Kategori'],
    'Nilai': [len(df), df['kategori'].nunique()]
})
print("TABEL: DATA DIMUAT (tanpa stopword removal)")
display(data_info)

TABEL: DATA DIMUAT (tanpa stopword removal)


,Keterangan,Nilai
0,Total Data Valid,295
1,Jumlah Kategori,7


## 3.1 Fine-Tuning IndoBERT dengan SimCSE (Unsupervised)
Metode **Unsupervised SimCSE** digunakan untuk meningkatkan kualitas representasi vektor kalimat.
- **Loss Function**: `MultipleNegativesRankingLoss`
- **Data Input**: `InputExample(texts=[text, text])` (Positive pair dari kalimat yang sama dengan dropout mask berbeda)
- **Epochs**: 3 (Unsupervised biasanya efisien dengan sedikit epoch)

In [38]:
# 2. Persiapan Data untuk SimCSE (Unsupervised)
# Input format: InputExample(texts=[text, text]) -> Model akan menganggap ini sebagai positive pair dengan dropout berbeda
train_examples = []
for text in df['deskripsi_clean'].tolist():
    train_examples.append(InputExample(texts=[text, text]))

# PENTING: Gunakan model.smart_batching_collate agar InputExample diproses dengan benar menjadi tensor
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=BATCH_SIZE, collate_fn=model.smart_batching_collate)
train_loss = losses.MultipleNegativesRankingLoss(model)

print(f"📊 Training Data: {len(train_examples)} pairs")

📊 Training Data: 295 pairs


In [39]:
# 3. Training Loop Manual (untuk visualisasi loss per epoch)
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

# Setup Optimizer & Scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
total_steps = len(train_dataloader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=int(total_steps * 0.1),
    num_training_steps=total_steps
)

print("="*80)
print("🚀 TRAINING INDOBERT dengan SimCSE + MultipleNegativesRankingLoss")
print("="*80)

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in progress_bar:
        # Batch dari smart_batching_collate sudah dalam format [features, labels]
        features, labels = batch
        
        # Forward pass melalui loss function
        loss_value = train_loss(features, labels)
        
        # Backward pass
        loss_value.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        
        total_loss += loss_value.item()
        
        # Opsional: Update progress bar dengan current loss
        progress_bar.set_postfix({'loss': f'{loss_value.item():.4f}'})
        
    avg_loss = total_loss / len(train_dataloader)
    print(f"   Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")

print("\n✅ Training Selesai!")

🚀 TRAINING INDOBERT dengan SimCSE + MultipleNegativesRankingLoss


Epoch 1/3: 100%|██████████| 37/37 [03:20<00:00,  5.42s/it, loss=0.0002]


   Epoch 1 - Average Loss: 0.0000


Epoch 2/3: 100%|██████████| 37/37 [03:03<00:00,  4.96s/it, loss=0.0000]


   Epoch 2 - Average Loss: 0.0000


Epoch 3/3: 100%|██████████| 37/37 [03:10<00:00,  5.15s/it, loss=0.0000]

   Epoch 3 - Average Loss: 0.0000

✅ Training Selesai!


In [40]:
# 3.1.2 Verifikasi Hasil Training (Sanity Check)
# Karena ini unsupervised, tabel loss mungkin kosong jika tidak ada evaluator.
# Mari kita cek secara kualitatif apakah model sudah paham konteks dengan mencari kemiripan.

from sentence_transformers import util

print("🔍 SANITY CHECK: Mencari wisata yang mirip dengan 'wisata alam air terjun'...")

# 1. Encode query
query = "wisata alam air terjun yang sejuk"
query_emb = model.encode(query, convert_to_tensor=True)

# 2. Encode corpus (sample 20 data pertama untuk cepat)
corpus_texts = df['deskripsi_clean'].tolist()
corpus_embs = model.encode(corpus_texts, convert_to_tensor=True)

# 3. Hitung similarity
hits = util.semantic_search(query_emb, corpus_embs, top_k=5)[0]

print(f"\nQuery: '{query}'")
print("Top 5 Recommendations:")
print("-" * 50)
for hit in hits:
    idx = hit['corpus_id']
    score = hit['score']
    name = df.iloc[idx]['nama']
    print(f"[{score:.4f}] {name}")
print("-" * 50)
print("Jika skor bervariasi dan relevan (misal banyak curug), maka embedding berhasil terbentuk.")

🔍 SANITY CHECK: Mencari wisata yang mirip dengan 'wisata alam air terjun'...

Query: 'wisata alam air terjun yang sejuk'
Top 5 Recommendations:
--------------------------------------------------
[0.3382] Gunung Peyek
[0.3189] Kolam Gunung Leutik Bolang Bogor
[0.2960] Curug Hordeng
[0.2897] Green Canyon Citamiang
[0.2854] Curug Bidadari
--------------------------------------------------
Jika skor bervariasi dan relevan (misal banyak curug), maka embedding berhasil terbentuk.


## 3.2 Generate Embeddings (with Fine-Tuned Model)

In [41]:
print("🚀 Generating embeddings with fine-tuned model...")
texts = df['deskripsi_clean'].tolist()

# SentenceTransformer membuat proses encoding menjadi sangat simpel
embeddings = model.encode(texts, batch_size=8, show_progress_bar=True, convert_to_numpy=True)

embedding_info = pd.DataFrame({
    'Statistik': ['Shape', 'Min', 'Max', 'Mean', 'Std'],
    'Nilai': [str(embeddings.shape), 
              f"{embeddings.min():.4f}", 
              f"{embeddings.max():.4f}",
              f"{embeddings.mean():.4f}",
              f"{embeddings.std():.4f}"]
})
print("\nTABEL: EMBEDDING STATISTICS")
display(embedding_info)

🚀 Generating embeddings with fine-tuned model...


Batches: 100%|██████████| 37/37 [00:18<00:00,  2.03it/s]


TABEL: EMBEDDING STATISTICS


,Statistik,Nilai
0,Shape,"(295, 768)"
1,Min,-2.1289
2,Max,2.0315
3,Mean,-0.0003
4,Std,0.4162


## 3.3 Simpan Embeddings

In [ ]:
# Simpan embeddings
np.save(f'{DATA_PATH}indobert_embeddings.npy', embeddings)

files_saved = pd.DataFrame({
    'File': ['indobert_embeddings.npy'],
    'Deskripsi': ['IndoBERT embeddings (768-dim)'],
    'Shape': [str(embeddings.shape)]
})
print("TABEL: FILES SAVED")
display(files_saved)

print("\n✅ INDOBERT EMBEDDING SELESAI!")

TABEL: FILES SAVED


,File,Deskripsi,Shape
0,indobert_embeddings.npy,IndoBERT embeddings (768-dim),"(295, 768)"



✅ INDOBERT EMBEDDING SELESAI!


: 